In [1]:
import gc
from pathlib import Path
import numpy as np
import pandas as pd


In [2]:
# ============================================================
# CONFIGURATION
# ============================================================

def find_project_root():
    """
    Find project root containing data/tables.
    Works whether notebook is launched from root or notebooks/.
    """
    current = Path.cwd().resolve()

    for path in [current, *current.parents]:
        if (path / "data" / "tables").exists():
            return path

    raise FileNotFoundError(
        "Could not find project root containing data/tables/"
    )


PROJECT_ROOT = find_project_root()

TABLES_DIR = PROJECT_ROOT / "data" / "tables"
PART1_DIR = TABLES_DIR / "part_1"
ABS_DIR = PROJECT_ROOT / "data" / "raw_abs"

CURATED_DIR = (
    PROJECT_ROOT
    / "data"
    / "curated"
    / "curated_transactions"
)

CURATED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Temporary/provisional fraud threshold
FRAUD_THRESHOLD = 80


print("PROJECT ROOT:", PROJECT_ROOT)
print("TABLES:", TABLES_DIR)
print("RAW ABS:", ABS_DIR)
print("CURATED OUTPUT:", CURATED_DIR)

PROJECT ROOT: /Users/DinadiPerera/Downloads/GitHub/MAST30034_Project_2
TABLES: /Users/DinadiPerera/Downloads/GitHub/MAST30034_Project_2/data/tables
RAW ABS: /Users/DinadiPerera/Downloads/GitHub/MAST30034_Project_2/data/raw_abs
CURATED OUTPUT: /Users/DinadiPerera/Downloads/GitHub/MAST30034_Project_2/data/curated/curated_transactions


In [3]:
# ============================================================
# PART 1 — LOAD
# ============================================================

def load_part1():

    consumer = pd.read_csv(
        PART1_DIR / "tbl_consumer.csv",
        sep="|"
    )

    consumer_details = pd.read_parquet(
        PART1_DIR / "consumer_user_details.parquet"
    )

    merchants = pd.read_parquet(
        PART1_DIR / "tbl_merchants.parquet"
    )

    consumer_fraud = pd.read_csv(
        PART1_DIR / "consumer_fraud_probability.csv"
    )

    merchant_fraud = pd.read_csv(
        PART1_DIR / "merchant_fraud_probability.csv"
    )

    return (
        consumer,
        consumer_details,
        merchants,
        consumer_fraud,
        merchant_fraud
    )

In [4]:
# ============================================================
# PART 1 — CLEAN
# ============================================================

def clean_consumer(df):
    df = df.copy()
    # Postcodes are identifiers, not measurements
    df["postcode"] = (
        pd.to_numeric(
            df["postcode"],
            errors="coerce"
        )
        .astype("Int64")
        .astype("string")
        .str.zfill(4)
    )
    return df


def clean_consumer_fraud(df):
    df = df.copy()
    df = df.drop_duplicates()
    # Fraud data is day-level
    df["order_datetime"] = pd.to_datetime(
        df["order_datetime"].astype(str),
        errors="raise"
    ).dt.normalize()
    df["fraud_probability"] = pd.to_numeric(
        df["fraud_probability"],
        errors="raise"
    ).astype(np.float32)
    return df


def clean_merchant_fraud(df):
    df = df.copy()
    df = df.drop_duplicates()
    df["order_datetime"] = pd.to_datetime(
        df["order_datetime"].astype(str),
        errors="raise"
    ).dt.normalize()
    df["fraud_probability"] = pd.to_numeric(
        df["fraud_probability"],
        errors="raise"
    ).astype(np.float32)
    return df


In [5]:
# ============================================================
# VALIDATION
# ============================================================

VALID_STATES = {
    "ACT", "NSW", "NT", "QLD",
    "SA", "TAS", "VIC", "WA"
}

VALID_GENDERS = {
    "Male", "Female", "Undisclosed"
}


def validate_consumer(df):

    assert df["consumer_id"].notna().all()
    assert df["consumer_id"].is_unique

    assert df["state"].isin(VALID_STATES).all()
    assert df["gender"].isin(VALID_GENDERS).all()

    assert df["postcode"].str.len().eq(4).all()

    assert df["name"].str.strip().ne("").all()
    assert df["address"].str.strip().ne("").all()


def validate_consumer_details(df):

    assert df["consumer_id"].notna().all()
    assert df["user_id"].notna().all()

    assert df["consumer_id"].is_unique
    assert df["user_id"].is_unique


def validate_merchants(df):

    assert df.index.notna().all()
    assert df.index.is_unique

    assert (
        pd.Series(df.index.astype(str))
        .str.len()
        .eq(11)
        .all()
    )

    assert df["name"].notna().all()
    assert df["name"].str.strip().ne("").all()


def validate_consumer_fraud(df):

    assert df["user_id"].notna().all()
    assert df["order_datetime"].notna().all()
    assert df["fraud_probability"].notna().all()

    assert df["fraud_probability"].between(
        0, 100
    ).all()

    assert df.duplicated().sum() == 0


def validate_merchant_fraud(df):

    assert df["merchant_abn"].notna().all()
    assert df["order_datetime"].notna().all()
    assert df["fraud_probability"].notna().all()

    assert df["fraud_probability"].between(
        0, 100
    ).all()

    assert pd.api.types.is_datetime64_any_dtype(
        df["order_datetime"]
    )


def validate_consumer_relationship(
    consumer,
    consumer_details
):

    consumer.merge(
        consumer_details,
        on="consumer_id",
        validate="one_to_one"
    )

    assert (
        set(consumer["consumer_id"])
        ==
        set(consumer_details["consumer_id"])
    )

In [6]:
# ============================================================
# EXTERNAL DATA — POSTCODE -> SA2
# ============================================================

def load_sa2_mapping():

    path = ABS_DIR / "poa_to_sa2.parquet"

    poa_to_sa2 = pd.read_parquet(path)

    poa_to_sa2["POA_CODE_2021"] = (
        pd.to_numeric(
            poa_to_sa2["POA_CODE_2021"],
            errors="coerce"
        )
        .astype("Int64")
        .astype("string")
        .str.zfill(4)
    )

    # Choose highest-ratio SA2 for each postcode
    poa_to_sa2_best = (
        poa_to_sa2
        .sort_values(
            "ratio",
            ascending=False
        )
        .drop_duplicates(
            subset="POA_CODE_2021",
            keep="first"
        )
    )

    return poa_to_sa2_best

In [7]:
# ============================================================
# EXTERNAL DATA — SEIFA & INCOME FEATURES
# ============================================================

def load_seifa_and_features():
    seifa_path = ABS_DIR / "SEIFA_2021_SA2.xlsx"
    seifa = pd.read_excel(
        seifa_path,
        sheet_name="Table 1",
        skiprows=6,
        header=None
    )

    seifa.columns = [
        "SA2_CODE_2021",
        "SA2_NAME_2021",
        "disadvantage_score",
        "disadvantage_decile",
        "adv_disadv_score",
        "adv_disadv_decile",
        "economic_resources_score",
        "economic_resources_decile",
        "education_occupation_score",
        "education_occupation_decile",
        "usual_resident_population"
    ]

    seifa["SA2_CODE_2021"] = pd.to_numeric(
        seifa["SA2_CODE_2021"],
        errors="coerce"
    ).astype("Int64")
    seifa = seifa[seifa["SA2_CODE_2021"].notna()].copy()

    numeric_cols = [
        "disadvantage_score",
        "disadvantage_decile",
        "adv_disadv_score",
        "adv_disadv_decile",
        "economic_resources_score",
        "economic_resources_decile",
        "education_occupation_score",
        "education_occupation_decile",
        "usual_resident_population"
    ]

    for col in numeric_cols:
        seifa[col] = pd.to_numeric(seifa[col], errors="coerce").astype(np.float32)

    seifa["SA2_NAME_2021"] = seifa["SA2_NAME_2021"].astype("string")

    # Load income/IRSAD features
    features_path = ABS_DIR / "loaded_features_df.parquet"
    features_df = pd.read_parquet(features_path)
    features_df["SA2_CODE_2021"] = pd.to_numeric(
        features_df["SA2_CODE_2021"],
        errors="coerce"
    ).astype("Int64")

    for col in features_df.columns:
        if col != "SA2_CODE_2021" and pd.api.types.is_numeric_dtype(features_df[col]):
            features_df[col] = features_df[col].astype(np.float32)

    # Merge SEIFA with Income features
    full_features = seifa.merge(
        features_df,
        on="SA2_CODE_2021",
        how="left",
        suffixes=("", "_income")
    )

    # State prefix mapping for ABS SA2 codes
    state_to_code = {
        '1': 'NSW', '2': 'VIC', '3': 'QLD', '4': 'SA',
        '5': 'WA', '6': 'TAS', '7': 'NT', '8': 'ACT', '9': 'OT'
    }

    feature_cols = [
        "disadvantage_score",
        "disadvantage_decile",
        "adv_disadv_score",
        "adv_disadv_decile",
        "economic_resources_score",
        "economic_resources_decile",
        "education_occupation_score",
        "education_occupation_decile",
        "IRSAD_score",
        "IRSAD_decile",
        "median_personal_income_weekly",
        "median_family_income_weekly",
        "median_household_income_weekly"
    ]

    # Calculate state-level averages for fallback imputation
    sa2_str = full_features["SA2_CODE_2021"].astype(str)
    state_series = sa2_str.str[0].map(state_to_code).rename("state")
    
    state_averages = (
        full_features
        .groupby(state_series)[feature_cols]
        .mean()
        .astype(np.float32)
    )

    return full_features, state_averages, feature_cols


In [8]:
# ============================================================
# FRAUD JOINS
# ============================================================

def add_consumer_fraud(
    transactions,
    consumer_fraud
):

    fraud = consumer_fraud[
        [
            "user_id",
            "order_datetime",
            "fraud_probability"
        ]
    ].rename(
        columns={
            "fraud_probability":
            "consumer_fraud_probability"
        }
    )

    return transactions.merge(
        fraud,
        on=[
            "user_id",
            "order_datetime"
        ],
        how="left",
        validate="many_to_one"
    )


def add_merchant_fraud(
    transactions,
    merchant_fraud
):

    fraud = merchant_fraud[
        [
            "merchant_abn",
            "order_datetime",
            "fraud_probability"
        ]
    ].rename(
        columns={
            "fraud_probability":
            "merchant_fraud_probability"
        }
    )

    return transactions.merge(
        fraud,
        on=[
            "merchant_abn",
            "order_datetime"
        ],
        how="left",
        validate="many_to_one"
    )

In [9]:
# ============================================================
# PARTS 2–4 — TRANSACTIONS
# ============================================================

def load_transaction_part(part):
    path = TABLES_DIR / part
    if not path.exists():
        raise FileNotFoundError(
            f"Missing transaction dataset: {path}"
        )
    print(f"Loading {part}...")
    return pd.read_parquet(path)


def clean_transactions(df):
    df = df.copy()
    df["order_datetime"] = pd.to_datetime(
        df["order_datetime"].astype(str),
        errors="raise"
    ).dt.normalize()
    if "dollar_value" in df.columns:
        df["dollar_value"] = df["dollar_value"].astype(np.float32)

    assert df["order_id"].notna().all()
    assert df["user_id"].notna().all()
    assert df["merchant_abn"].notna().all()
    assert df["order_id"].is_unique, (
        "Duplicate transaction order IDs detected"
    )
    return df


In [10]:
# ============================================================
# IMPUTATION & APPROXIMATION FLAGS (MEMORY-OPTIMIZED)
# ============================================================

def impute_missing_features(df, state_averages, feature_cols):
    # Perform in-place imputation to prevent duplicating 3.6M+ rows in RAM
    for col in feature_cols:
        if col in df.columns:
            # Compact boolean approximation flag
            df[f"{col}_is_approximated"] = df[col].isna().astype("bool")

            # Fill missing entries with state average in-place
            state_mapping = df["state"].map(state_averages[col]).astype(np.float32)
            df[col] = df[col].fillna(state_mapping).astype(np.float32)

    return df


In [11]:
# ============================================================
# MASTER BATCH ETL
# ============================================================

def build_curated_transactions():
    print("=== BNPL BATCH ETL START ===")

    # --------------------------------------------------------
    # 1. LOAD PART 1
    # --------------------------------------------------------
    (
        consumer,
        consumer_details,
        merchants,
        consumer_fraud,
        merchant_fraud
    ) = load_part1()

    # --------------------------------------------------------
    # 2. CLEAN PART 1
    # --------------------------------------------------------
    consumer = clean_consumer(consumer)
    consumer_fraud = clean_consumer_fraud(consumer_fraud)
    merchant_fraud = clean_merchant_fraud(merchant_fraud)

    # --------------------------------------------------------
    # 3. VALIDATE PART 1
    # --------------------------------------------------------
    validate_consumer(consumer)
    validate_consumer_details(consumer_details)
    validate_merchants(merchants)
    validate_consumer_fraud(consumer_fraud)
    validate_merchant_fraud(merchant_fraud)
    validate_consumer_relationship(consumer, consumer_details)

    # --------------------------------------------------------
    # 4. PREPARE LOOKUP TABLES
    # --------------------------------------------------------
    consumer_enriched = consumer.merge(
        consumer_details,
        on="consumer_id",
        how="left",
        validate="one_to_one"
    )

    merchants = merchants.reset_index()
    sa2_mapping = load_sa2_mapping()
    seifa, state_averages, feature_cols = load_seifa_and_features()

    # --------------------------------------------------------
    # 5. PROCESS TRANSACTION PARTITIONS SEPARATELY
    # --------------------------------------------------------
    total_rows = 0

    for part in ["part_2", "part_3", "part_4"]:
        output_file = CURATED_DIR / f"{part}.parquet"

        # Skip already created files
        if output_file.exists():
            print(f"\n{part} already exists — skipping.")
            continue

        print("\n" + "=" * 50)
        print(f"PROCESSING {part}")
        print("=" * 50)

        # Load and clean transaction part
        transactions = load_transaction_part(part)
        transactions = clean_transactions(transactions)
        original_rows = len(transactions)
        print(f"{part} transactions loaded: {original_rows:,}")

        # Consumer join
        curated = transactions.merge(
            consumer_enriched,
            on="user_id",
            how="left",
            validate="many_to_one"
        )
        del transactions
        gc.collect()

        # Merchant join
        curated = curated.merge(
            merchants,
            on="merchant_abn",
            how="left",
            validate="many_to_one",
            suffixes=("_consumer", "_merchant")
        )
        gc.collect()

        # Consumer & Merchant fraud joins
        curated = add_consumer_fraud(curated, consumer_fraud)
        curated = add_merchant_fraud(curated, merchant_fraud)

        # Fraud flags
        curated["consumer_is_fraud"] = (
            curated["consumer_fraud_probability"] >= FRAUD_THRESHOLD
        ).astype("bool")

        curated["merchant_is_fraud"] = (
            curated["merchant_fraud_probability"] >= FRAUD_THRESHOLD
        ).astype("bool")

        # Postcode -> SA2
        curated["postcode"] = (
            curated["postcode"].astype("string").str.zfill(4)
        )

        curated = curated.merge(
            sa2_mapping[[
                "POA_CODE_2021", "SA2_CODE_2021",
                "mb_count", "poa_total", "ratio"
            ]],
            left_on="postcode",
            right_on="POA_CODE_2021",
            how="left",
            validate="many_to_one"
        )
        gc.collect()

        # SEIFA & Income features
        curated["SA2_CODE_2021"] = pd.to_numeric(
            curated["SA2_CODE_2021"],
            errors="coerce"
        ).astype("Int64")

        curated = curated.merge(
            seifa,
            on="SA2_CODE_2021",
            how="left",
            validate="many_to_one"
        )
        gc.collect()

        # In-place feature imputation & flags
        curated = impute_missing_features(curated, state_averages, feature_cols)
        gc.collect()

        # Validation check
        if len(curated) != original_rows:
            raise ValueError(
                f"{part}: transaction count changed. "
                f"Original={original_rows:,}, Final={len(curated):,}"
            )

        assert curated["order_id"].is_unique, (
            f"{part}: duplicate order IDs detected"
        )

        # Summary logs
        print(f"Final transactions: {len(curated):,}")
        print("Consumer fraud matched:", f"{curated['consumer_fraud_probability'].notna().sum():,}")
        print("Merchant fraud matched:", f"{curated['merchant_fraud_probability'].notna().sum():,}")
        print("Missing SA2:", f"{curated['SA2_CODE_2021'].isna().sum():,}")
        print("Imputed IRSAD features:", f"{curated['IRSAD_score_is_approximated'].sum():,}")

        # Save Parquet with row group chunking to minimize write RAM buffer
        print(f"Saving {part}...")
        curated.to_parquet(
            output_file,
            index=False,
            engine="pyarrow",
            row_group_size=250_000,
            compression="snappy"
        )

        print(f"{part} SAVED ✓")
        print(output_file)
        total_rows += len(curated)

        # Free memory before next partition
        del curated
        gc.collect()
        print(f"{part} cleared from RAM.")

    print("\n=== BNPL BATCH ETL COMPLETE ===")
    print("Curated dataset directory:", CURATED_DIR)
    return CURATED_DIR


In [ ]:
CURATED_DATA_PATH = (
    build_curated_transactions()
)